# Ramping juxta analysis

In [1]:
import workbench
from scipy.io import loadmat, savemat
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import sqlite3
import os
import re

In [2]:
# fetch the database entries
database_path = Path(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT r.Animal_Id, r.Cell_Id, r.Folderpath, r.Condition, r.exp_type
FROM Recordings as r
WHERE exp_type = 'juxta'
AND r.Condition = 'ramp'
AND USE = 1
ORDER BY r.Cell_Id DESC"""

conn = sqlite3.connect(database_path)
db = pd.read_sql_query(sql, conn)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, find_peaks
# parquet backend requires pyarrow installed


########################################
# 1. schema + serialization helpers
########################################

EXPECTED_COLS = [
    "Animal_Id", "Condition", "Cell_Id", "nstims",
    "RasterTimes", "RasterRows", "RasterRate",
    "pupil_psth", "whisk_psth", "all_whisk",
    "mot_avg", "trigger_time",
]

LISTY_COLS = [
    "RasterTimes", "RasterRows", "RasterRate",
    "pupil_psth", "whisk_psth", "all_whisk",
]

def _to_list_or_none(x):
    """Convert values (None, ndarray, list of ndarrays, etc.) into
    parquet-friendly Python lists or None.
    """
    if x is None:
        return None
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (list, tuple)):
        out = []
        for v in x:
            if isinstance(v, np.ndarray):
                out.append(v.tolist())
            elif isinstance(v, (list, tuple)):
                out.append(list(v))
            else:
                out.append(v)
        return out
    return x  # scalar etc.

def _make_empty_row():
    """Defaults for when data (like pupil) is missing but
    we still want the column present.
    """
    return {
        "Animal_Id": None,
        "Condition": None,
        "Cell_Id": None,
        "nstims": 0,
        "RasterTimes": [],
        "RasterRows": [],
        "RasterRate": [],
        "pupil_psth": [],
        "whisk_psth": [],
        "all_whisk": [],
        "mot_avg": None,
        "trigger_time": None,
    }

def _finalize_row(row_dict):
    """Ensure all columns exist and convert
    nested arrays into parquet-friendly lists.
    """
    # fill any missing columns
    defaults = _make_empty_row()
    for c in EXPECTED_COLS:
        if c not in row_dict:
            row_dict[c] = defaults[c]

    # normalize ragged/nested
    for c in LISTY_COLS:
        row_dict[c] = _to_list_or_none(row_dict[c])

    row_dict["mot_avg"] = _to_list_or_none(row_dict["mot_avg"])
    row_dict["trigger_time"] = _to_list_or_none(row_dict["trigger_time"])

    return row_dict


########################################
# 2. signal helpers used in tonestim_output_ramp
########################################

def _normalize_range(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    xmin = np.nanmin(x)
    xmax = np.nanmax(x)
    if xmax == xmin:
        return np.zeros_like(x)
    return (x - xmin) / (xmax - xmin)

def _smooth_moving_average(x, window_size):
    # approximate MATLAB smooth(x, N) with reflective-pad moving average
    w = int(window_size)
    if w < 1:
        return np.asarray(x, dtype=float)
    if w % 2 == 0:
        w += 1
    x = np.asarray(x, dtype=float).ravel()
    pad = w // 2
    xpad = np.pad(x, pad, mode='reflect')
    kernel = np.ones(w) / w
    y = np.convolve(xpad, kernel, mode='valid')
    return y

def _but_filter_bandpass(x, order, low_high_norm):
    # Butterworth bandpass, filtfilt like MATLAB's ButFilter(...,'bandpass')
    low, high = low_high_norm
    b, a = butter(order, [low, high], btype='bandpass')
    return filtfilt(b, a, x, method="gust")


########################################
# 3. MATLAB helper ports
########################################

def correctKeyboardTimes(keyboard_times, sound_wave):
    """
    Port of MATLAB correctKeyboardTimes.m
    keyboard_times: 1D array (sec)
    sound_wave: 1D array (voltage-like)
    returns corrected times in sec
    """
    keyboard_times = np.asarray(keyboard_times, dtype=float).ravel()
    sound_wave = np.asarray(sound_wave, dtype=float).ravel()

    diff_wave = np.abs(np.diff(sound_wave))
    peaks, info = find_peaks(diff_wave, height=0.04)
    peaktimes = (peaks - 1) / 25000.0  # 25 kHz sampling rate

    correctedKeyboardTimes = np.zeros_like(keyboard_times)
    for i, kt in enumerate(keyboard_times):
        if peaktimes.size == 0:
            correctedKeyboardTimes[i] = kt + 0.002
            continue
        diffs = np.abs(peaktimes - kt)
        idx = np.argmin(diffs)
        val = diffs[idx]
        if val > 0.01:  # >10 ms away
            correctedKeyboardTimes[i] = kt + 0.002
        else:
            correctedKeyboardTimes[i] = peaktimes[idx]
    return correctedKeyboardTimes

def TriggRaster(triggers, spk_times, SR=25000, halftime=0.06, nbins=100,
                doShuffle=False, nshuffles=500):
    """
    Port of MATLAB TriggRaster.m
    triggers: trigger indices in samples
    spk_times: spike times in seconds
    SR: sampling rate
    halftime: window +/- (sec)
    nbins: histogram bins
    """
    triggers = np.rint(np.asarray(triggers, dtype=float)).astype(int)
    spk_times = np.asarray(spk_times, dtype=float).ravel()
    spkSamples = np.rint(spk_times * float(SR)).astype(int)
    halfsamples = int(np.rint(float(halftime) * float(SR)))

    ntrials = int(triggers.size)
    ncols = (halfsamples * 2) + 1
    Chunks = np.zeros((ntrials, ncols), dtype=bool)

    for tt in range(ntrials):
        start = triggers[tt] - halfsamples
        stop = triggers[tt] + halfsamples
        chunk_samples = np.arange(start, stop + 1, dtype=int)
        Chunks[tt, :] = np.isin(chunk_samples, spkSamples)

    timechunk = np.linspace(-float(halftime), float(halftime), ncols)
    rows, cols = np.nonzero(Chunks)

    # PSTH bins
    col_1b = cols + 1  # emulate MATLAB 1-based indexing
    edges = np.linspace(1, ncols, nbins)
    counts, edges_used = np.histogram(col_1b, bins=edges)

    # MATLAB medfilt1(...,2) behavior: we approximate with midpoints
    edges2plot = (edges_used[:-1] + edges_used[1:]) / 2.0
    idx = np.rint(edges2plot).astype(int) - 1
    idx = np.clip(idx, 0, ncols - 1)

    if idx.size >= 2:
        time2scale = np.median(np.diff(timechunk[idx]))
    else:
        time2scale = (timechunk[-1] - timechunk[0]) / max(nbins - 1, 1)

    rate = counts.astype(float) / (max(ntrials, 1) * float(time2scale))

    if doShuffle:
        rng = np.random.default_rng(0)
        rateShuffle = np.zeros((nshuffles, counts.size), dtype=float)
        for sh in range(nshuffles):
            rand_shifts = rng.integers(low=-ncols, high=ncols + 1, size=ntrials)
            shuffMatrix = np.zeros_like(Chunks)
            for nr in range(ntrials):
                shuffMatrix[nr, :] = np.roll(Chunks[nr, :], shift=rand_shifts[nr])
            _, colS = np.nonzero(shuffMatrix)
            colS_1b = colS + 1
            countsS, _ = np.histogram(colS_1b, bins=edges_used)
            rateShuffle[sh, :] = countsS.astype(float) / float(time2scale)
    else:
        rateShuffle = np.empty((0, counts.size), dtype=float)

    RasterOut = {
        "raster_times": (timechunk[cols] * 1000.0).astype(float),
        "raster_rows": rows.astype(int),
        "rate": rate.astype(float),
        "time": np.rint(timechunk[idx] * 1000.0).astype(int),
        "matrix": Chunks.astype(bool),
        "rateShuff": rateShuffle,
        "time_bin": float(time2scale) * 1000.0,
    }
    return RasterOut


########################################
# 4. tonestim_output_ramp
########################################

def tonestim_output_ramp(data_row):
    """
    Port of tonestim_output_ramp.m (ramp condition), returning a
    1-row pandas DataFrame. Uses the schema helpers so all columns
    exist even if certain data (like pupil) are missing.
    """

    # constants
    tone_letter = ['a', 'w', 'e', 't']
    pupil_sr = 50
    halftime_spikes = 8        # sec window for TriggRaster
    rasterBins = halftime_spikes * 800
    halftime_mot = 8           # sec around trigger for pupil/whisk
    smooth_pupil_value = 15    # for smoothing pupil signal
    mot_samples = halftime_mot * 2 * pupil_sr

    print(f"{'':-^100}")
    print(f" Animal_Id {data_row['Animal_Id']}; Cell_Id {int(data_row['Cell_Id'])} being processed ")
    print(f"{'':-^100}")

    # init shared vars
    tone_onset = None
    tone_code = None
    RasterTimes = None
    RasterRows = None
    RasterRate = None
    pupil_psth = None
    whisk_psth = None
    all_whisk = None
    mot_avg = None
    trigger_time = None

    ########################################################
    # load trigger/ephys info for juxta OR behav experiment
    ########################################################

    if data_row['exp_type'] == 'juxta':
        # load exp_data.mat
        try:
            exp = loadmat(
                os.path.join(data_row['Folderpath'], 'exp_data.mat'),
                struct_as_record=False,
                squeeze_me=True
            )
        except Exception:
            print(f"Missing exp_data.mat for {data_row['Animal_Id']} Cell_Id {data_row['Cell_Id']}")
            # we'll still build an empty row at the end
            spkT = None
        else:
            processed_data = exp.get('processed_data', None)
            raw_data = exp.get('raw_data', None)

            if processed_data is None or raw_data is None:
                print(f"Cell ID = {data_row['Cell_Id']} processed_data/raw_data missing")
                spkT = None
            else:
                ssd = getattr(processed_data, 'spike_sorting_data', None)
                if ssd is None:
                    print(f"WARNING: Cell ID = {data_row['Cell_Id']} processed_data EMPTY")
                    spkT = None
                else:
                    spkT = getattr(ssd, 'spike_times', None)

                ephys = getattr(raw_data, 'ephys_data', None)
                if ephys is not None:
                    tone_times = getattr(ephys, 'keyboard_times', None)
                    tone_code = getattr(ephys, 'keyboard_codes', None)
                    ephys_sr = int(np.round(getattr(ephys, 'sampling_rate', 25000)))
                    ephy_times = getattr(ephys, 'ephy_times', None)

                    if tone_times is not None and ephy_times is not None:
                        tone_onset = np.zeros(len(np.atleast_1d(tone_times)), dtype=int)
                        for ss, t in enumerate(np.atleast_1d(tone_times)):
                            idx = np.argmin(np.abs(ephy_times - t))
                            tone_onset[ss] = idx
                else:
                    ephys_sr = 25000  # fallback

        # build rasters only if we have spikes + code
        if (tone_onset is not None and tone_code is not None and
            isinstance(spkT, (list, np.ndarray))):
            RasterTimes = []
            RasterRows = []
            RasterRate = []

            for letter in tone_letter:
                this_code = ord(letter)
                # tone_code assumed shape (N,1) or (N,)
                tcode_col = tone_code[:, 0] if (
                    hasattr(tone_code, "ndim") and tone_code.ndim > 1
                ) else tone_code
                tcode_int = tcode_col.astype(int) if np.issubdtype(
                    np.asarray(tcode_col).dtype, np.number
                ) else np.array([ord(c) for c in np.atleast_1d(tcode_col)])

                mask = np.isin(tcode_int, this_code)
                tone_triggers = tone_onset[mask]

                raster = TriggRaster(
                    triggers=tone_triggers,
                    spk_times=spkT,
                    SR=ephys_sr,
                    halftime=halftime_spikes,
                    nbins=rasterBins,
                    doShuffle=False,
                    nshuffles=500
                )
                RasterTimes.append(raster["raster_times"])
                RasterRows.append(raster["raster_rows"])
                RasterRate.append(raster["rate"])

    elif data_row['exp_type'] == 'behav':
        # replicate the logic of the MATLAB 'behav' branch:
        variables = ["*Ch31", "*Ch5", "*Ch6"]
        pattern_ch6 = r'^Data\d*\w*_Ch6'
        filename = data_row['Filename'] + '.mat'
        beh_path = os.path.join(data_row['Folderpath'], filename)

        data = loadmat(beh_path, struct_as_record=False, squeeze_me=True)
        fields = list(data.keys())
        matching_fields = [f for f in fields if re.search(pattern_ch6, f)]

        pupil_begins = None

        if 'Ch6' in data:
            ch6 = data['Ch6']
            ch6_vals = np.asarray(getattr(ch6, 'values', []), dtype=float)
            pupilttlvals = np.diff(ch6_vals)
            peaks, info = find_peaks(pupilttlvals, height=3)
            if peaks.size > 0:
                idx = peaks[0]
                times6 = np.asarray(getattr(ch6, 'times', []), dtype=float)
                pupil_begins = times6[idx] if idx < times6.size else None

            ch31 = data.get('Ch31', None)
            whitelist = np.array(list('awertzuikp'))

            if ch31 is not None:
                c_codes = np.asarray(getattr(ch31, 'codes', []))
                c_times = np.asarray(getattr(ch31, 'times', []), dtype=float)
                firstcol = c_codes[:, 0] if (
                    hasattr(c_codes, "ndim") and c_codes.ndim > 1
                ) else c_codes
                mask = np.isin(
                    np.char.array(firstcol.astype(np.uint8)).astype('<U1'),
                    whitelist
                )
                tone_code = firstcol[mask]
                tone_times = c_times[mask]
            else:
                tone_code = None
                tone_times = None

            ch5 = data.get('Ch5', None)
            if ch5 is not None and tone_times is not None and pupil_begins is not None:
                ch5_vals = np.asarray(getattr(ch5, 'values', []), dtype=float)
                corrected = correctKeyboardTimes(tone_times, ch5_vals)
                tone_onset = corrected - pupil_begins

        elif len(matching_fields) > 0:
            ch6name = matching_fields[0]
            ch6 = data[ch6name]
            ch6_vals = np.asarray(getattr(ch6, 'values', []), dtype=float)
            pupilttlvals = np.diff(ch6_vals)
            peaks, info = find_peaks(pupilttlvals, height=3)
            if peaks.size > 0:
                idx = peaks[0]
                times6 = np.asarray(getattr(ch6, 'times', []), dtype=float)
                pupil_begins = times6[idx] if idx < times6.size else None

            pattern31 = r'^Data\d*\w*_Ch31'
            pattern5 = r'^Data\d*\w*_Ch5'
            patter31match = [f for f in fields if re.search(pattern31, f)]
            pattern5match = [f for f in fields if re.search(pattern5, f)]

            whitelist = np.array(list('awertzuikp'))
            if patter31match:
                ch31 = data[patter31match[0]]
                c_codes = np.asarray(getattr(ch31, 'codes', []))
                c_times = np.asarray(getattr(ch31, 'times', []), dtype=float)
                firstcol = c_codes[:, 0] if (
                    hasattr(c_codes, "ndim") and c_codes.ndim > 1
                ) else c_codes
                mask = np.isin(
                    np.char.array(firstcol.astype(np.uint8)).astype('<U1'),
                    whitelist
                )
                tone_code = firstcol[mask]
                tone_times = c_times[mask]
            else:
                tone_code = None
                tone_times = None

            if pattern5match and tone_times is not None and pupil_begins is not None:
                ch5 = data[pattern5match[0]]
                ch5_vals = np.asarray(getattr(ch5, 'values', []), dtype=float)
                corrected = correctKeyboardTimes(tone_times, ch5_vals)
                tone_onset = corrected - pupil_begins

        # no raster calc in 'behav' branch in your MATLAB version,
        # so Raster* remain None here.

    # nstims (works for both juxta & behav)
    nstims_val = int(0 if tone_onset is None else len(np.atleast_1d(tone_onset)))

    ########################################################
    # pupil / motion section (exists for both exp_type)
    ########################################################
    # NOTE: the original MATLAB code uses pupil_data.mat with fields:
    #   pupil_out.motion
    #   pupil_out.pupil_area
    #   pupil_out.pupil_times
    #   pupil_out.sr
    #   pupil_out.correction
    # We replicate that logic here.

    ephys_sr = 25000  # MATLAB hardcoded this before pupil processing
    pout_path = os.path.join(data_row['Folderpath'], 'pupil_data.mat')

    print(pout_path)
    try:
        pout = loadmat(pout_path, struct_as_record=False, squeeze_me=True)
    except Exception:
        # silently skip pupil data (like MATLAB catch+return early)
        print('inside exception')
        pout = None

    if pout is not None and 'pupil_out' in pout:
        pupil_out = pout['pupil_out']
        # safe attribute/field getter for MATLAB struct-like objects

        if pupil_times is not None and not (isinstance(correction, str) and correction == 'aligned'):
            whisk_motion = np.asarray(g(pupil_out, 'motion', []), dtype=float)
            pupil_area = np.asarray(g(pupil_out, 'pupil_area', []), dtype=float)
            sr = float(g(pupil_out, 'sr', pupil_sr))
            pupil_times = np.asarray(pupil_times, dtype=float)

            # whisk artifact removal
            if whisk_motion.size > 0:
                max_min = np.nanmax(whisk_motion) - np.nanmin(whisk_motion)
                stdv = np.nanstd(whisk_motion)
                mm_std_ratio = np.inf if stdv == 0 else (max_min / stdv)
                if mm_std_ratio > 7:
                    clip_thr = np.nanpercentile(whisk_motion, 99.7)
                    whisk_motion = whisk_motion.copy()
                    whisk_motion[whisk_motion > clip_thr] = np.nan

            # smooth + bandpass pupil
            if pupil_area.size > 0:
                pupil_area = _smooth_moving_average(pupil_area, smooth_pupil_value)
                pupil_area = _but_filter_bandpass(
                    pupil_area,
                    order=3,
                    low_high_norm=(0.1 / (sr / 2.0), 1.0 / (sr / 2.0))
                )

            # normalize to [0,1]
            if pupil_area.size > 0:
                pupil_area = _normalize_range(pupil_area)
            if whisk_motion.size > 0:
                whisk_motion = _normalize_range(whisk_motion)

            # allocate outputs
            mot_avg = np.zeros((len(tone_letter), mot_samples, 2), dtype=float)
            pupil_psth = [None] * len(tone_letter)
            whisk_psth = [None] * len(tone_letter)
            all_whisk = [None] * len(tone_letter)

            # loop tones for pupil chunks
            for tt, letter in enumerate(tone_letter):
                if tone_code is None or tone_onset is None:
                    # no triggers -> empty
                    pupil_chunks = np.zeros((0, mot_samples))
                    whisk_chunks = np.zeros((0, mot_samples))
                    mot_avg[tt, :, 0] = np.nan
                    mot_avg[tt, :, 1] = np.nan
                    all_whisk[tt] = whisk_chunks
                    pupil_psth[tt] = pupil_chunks
                    whisk_psth[tt] = whisk_chunks
                    continue

                this_code = ord(letter)
                # extract tone triggers for this code
                tcode_col = tone_code[:, 0] if (
                    hasattr(tone_code, "ndim") and tone_code.ndim > 1
                ) else tone_code
                # convert to ints
                if np.issubdtype(np.asarray(tcode_col).dtype, np.number):
                    tcode_int = tcode_col.astype(int)
                else:
                    tcode_int = np.array([ord(c) for c in np.atleast_1d(tcode_col)])

                mask = np.isin(tcode_int, this_code)
                tone_triggers = np.atleast_1d(tone_onset)[mask]

                # trigger units: if huge numbers -> samples, convert to sec
                if tone_triggers.size > 0 and np.nanmax(tone_triggers) > 100000:
                    tone_triggers = tone_triggers / float(ephys_sr)

                # allocate chunks
                ntr = int(tone_triggers.size)
                pupil_chunks = np.full((ntr, mot_samples), np.nan, dtype=float)
                whisk_chunks = np.full((ntr, mot_samples), np.nan, dtype=float)

                # fill per stimulus
                for st in range(ntr):
                    t0 = float(tone_triggers[st])
                    chunk_win = (t0 - halftime_mot, t0 + halftime_mot)

                    if pupil_times.size == 0:
                        continue
                    if chunk_win[0] < 0 or chunk_win[1] > pupil_times[-1]:
                        continue

                    my_window = np.where(
                        (pupil_times >= chunk_win[0]) &
                        (pupil_times <= chunk_win[1])
                    )[0]

                    # fix off-by-one like MATLAB
                    if my_window.size == mot_samples - 1:
                        last = my_window[-1] + 1
                        if last < pupil_times.size:
                            my_window = np.concatenate([my_window, [last]])

                    if my_window.size > mot_samples:
                        continue

                    # grab data
                    pup_arr = pupil_area.ravel()[my_window] if pupil_area.ndim > 0 else []
                    wh_arr = whisk_motion.ravel()[my_window] if whisk_motion.ndim > 0 else []

                    if pup_arr.size == mot_samples:
                        pupil_chunks[st, :] = pup_arr
                    if wh_arr.size == mot_samples:
                        whisk_chunks[st, :] = wh_arr

                # time axis for motion/pupil
                trigger_time = np.linspace(-halftime_mot, halftime_mot, mot_samples)

                pupil_psth[tt] = pupil_chunks
                whisk_psth[tt] = whisk_chunks
                mot_avg[tt, :, 0] = np.nanmean(pupil_chunks, axis=0)
                mot_avg[tt, :, 1] = np.nanmean(whisk_chunks, axis=0)
                all_whisk[tt] = whisk_chunks

    ########################################################
    # package final row
    ########################################################

    row_out = {
        "Animal_Id": data_row["Animal_Id"],
        "Condition": data_row["Condition"],
        "Cell_Id": int(data_row["Cell_Id"]),
        "nstims": nstims_val,
        "RasterTimes": RasterTimes,
        "RasterRows": RasterRows,
        "RasterRate": RasterRate,
        "pupil_psth": pupil_psth,
        "whisk_psth": whisk_psth,
        "all_whisk": all_whisk,
        "mot_avg": mot_avg,
        "trigger_time": trigger_time,
    }

    row_out = _finalize_row(row_out)
    comb_table = pd.DataFrame([row_out], columns=EXPECTED_COLS)
    return comb_table


########################################
# 5. create_comb_table (ramp only, parquet)
########################################

def create_comb_table(data_table: pd.DataFrame, datapath: str, filename: str) -> pd.DataFrame:
    """
    Loops over the entire database (data_table),
    runs tonestim_output_ramp on rows where Condition == 'ramp',
    accumulates into a DataFrame with a stable schema,
    and saves to parquet.
    """

    rows_accum = []

    for i in range(len(data_table)):
        row = data_table.iloc[i]

        if str(row.get('Condition', '')).lower() == 'ramp':
            out_df = tonestim_output_ramp(row)  # 1-row df with EXPECTED_COLS
            rows_accum.append(out_df.iloc[0].to_dict())

    if rows_accum:
        comb_table = pd.DataFrame(rows_accum, columns=EXPECTED_COLS)
    else:
        # empty DataFrame with the right columns
        comb_table = pd.DataFrame([_make_empty_row()], columns=EXPECTED_COLS).iloc[0:0]

    save_path = os.path.join(datapath, f"{filename}.parquet")
    comb_table.to_parquet(save_path, index=False)
    print(f"Saved combined table to {save_path}")

    return comb_table


########################################
# 6. "load if exists else create" wrapper for ramp
########################################

def get_ramp_comb_table(datapath: str) -> pd.DataFrame:
    """
    Try loading the cached ramp parquet.
    If it doesn't exist, build it using ramp_juxta() and save.
    """
    parquet_file = os.path.join(datapath, 'ramp-juxta-pairs.parquet')

    try:
        ramp_comb_table = pd.read_parquet(parquet_file)
        print(f"Loaded existing table from {parquet_file}")
    except FileNotFoundError:
        print('-' * 150)
        print(' Table either non existing or missing. - Creation with this run '.center(150, '-'))
        print('-' * 150)

        # You provide this: must return the full DB as a DataFrame
        # with Animal_Id, Condition, Cell_Id, exp_type, Folderpath, etc.
        database_path = Path(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
        sql = """
        SELECT r.Animal_Id, r.Cell_Id, r.Folderpath, r.Condition, r.exp_type
        FROM Recordings as r
        WHERE exp_type = 'juxta'
        AND r.Condition = 'ramp'
        AND USE = 1
        ORDER BY r.Cell_Id DESC"""

        conn = sqlite3.connect(database_path)
        db = pd.read_sql_query(sql, conn)

        ramp_comb_table = create_comb_table(
            data_table=db,
            datapath=datapath,
            filename='ramp-juxta-pairs'
        )

    return ramp_comb_table



In [17]:

########################################
# 7. usage example (for context)
########################################
df = get_ramp_comb_table(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\comb_tables")

Loaded existing table from \\172.25.250.112\burgalossi\lab share\Data\Florian\comb_tables\ramp-juxta-pairs.parquet


In [18]:
p = df['RasterTimes']